In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kaushiksuresh147/bitcoin-tweets/Bitcoin_tweets_dataset_2.csv
/kaggle/input/datasets/kaushiksuresh147/bitcoin-tweets/Bitcoin_tweets.csv
/kaggle/input/datasets/tuilathai/cleand-text/btc_tweet_1.csv


In [ ]:
col_to_use = ['user_name','user_verified','user_favourites','date','text','hashtags', 'is_retweet']

df0 = pd.read_csv('/kaggle/input/datasets/kaushiksuresh147/bitcoin-tweets/Bitcoin_tweets.csv',
                  chunksize=100000,
                  usecols=col_to_use,
                  engine='python',
                  on_bad_lines='skip')
df = pd.concat(df0, ignore_index=True)

In [ ]:
df.head(10)

,user_name,user_favourites,user_verified,date,text,hashtags,is_retweet
0,DeSota Wilson,4838,False,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,['bitcoin'],False
1,CryptoND,25483,False,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""...","['Thursday', 'Btc', 'wallet', 'security']",False
2,Tdlmatias,924,False,2021-02-10 23:54:48,"Guys evening, I have read this article about B...",NaN,False
3,Crypto is the future,14,False,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,"['Bitcoin', 'FX', 'BTC', 'crypto']",False
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,10482,False,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,['BTC'],False
5,ZerrBenz™ ⚔ ✪ 20732,2444,False,2021-02-10 23:53:30,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,"['Crypto', 'Binance', 'Cashback']",False
6,Bitcoin-Bot,5728,False,2021-02-10 23:53:17,&lt;'fire' &amp; 'man'&gt;\n#Bitcoin #Crypto #...,"['Bitcoin', 'Crypto', 'BTC']",False
7,Cryptocurrencies / EUR,9,False,2021-02-10 23:52:42,🔄 Prices update in $EUR (1 hour):\n\n$BTC - ...,NaN,False
8,Mikcoin,238,False,2021-02-10 23:52:25,#BTC #Bitcoin #Ethereum #ETH #Crypto #cryptotr...,"['BTC', 'Bitcoin', 'Ethereum', 'ETH', 'Crypto'...",False
9,DeSota Wilson,4838,False,2021-02-10 23:52:08,.@Tesla’s #bitcoin investment is revolutionary...,"['bitcoin', 'crypto']",False


In [ ]:
df.tail(10)

,user_name,user_favourites,user_verified,date,text,hashtags,is_retweet
4693081,世界杯下注,0.0,False,2023-01-06 17:46:53,Jonas Cumberland #btc #彩票 Merle Noyes #世界杯直播 h...,"['btc', '彩票', '世界杯直播']",False
4693082,Gale Browning,55.0,False,2023-01-06 17:46:50,❤️ Join me at Bybit and earn exclusive rewards...,"['BTC', 'Airdrops', 'USDT', 'BYBIT', 'BANK', '...",False
4693083,Jeanie Phillips,64.0,False,2023-01-06 17:46:50,❤️ Join me at Bybit and earn exclusive rewards...,"['BTC', 'Airdrops', 'USDT', 'BYBIT', 'BANK', '...",False
4693084,世界杯投注平台,2.0,False,2023-01-06 17:46:44,Lorraine Harvey #btc #彩票 Coral Bart #世界杯直播 htt...,"['btc', '彩票', '世界杯直播']",False
4693085,Behzad.moni,1524.0,False,2023-01-06 17:46:36,#DogelonMars is the future. #TSUKA is the nex...,"['DogelonMars', 'TSUKA', 'SHIB', 'SAFEMOON', '...",False
4693086,TAnotepad,27466.0,False,2023-01-06 17:46:35,"Bitcoin squeeze is SUPER TIGHT, which way will...","['BTC', 'bitcoin', 'Crypto', 'cryptocurrency',...",False
4693087,Boba-Feh,125.0,False,2023-01-06 17:46:29,Closed #BTC short at 16725. Missed my long pla...,['BTC'],False
4693088,Ethereum Yoda,0.0,False,2023-01-06 17:46:22,#Ethereum price update: \n\n#ETH $1263.59 USD\...,"['Ethereum', 'ETH', 'Bitcoin', 'BTC', 'altcoin...",False
4693089,Bitcoin Price Ticker,9.0,False,2023-01-06 17:46:20,1₿ = $16814.7 -0.07%🔻\n\nDetails:\nChange: 🔻-1...,"['bitcoin', 'btc']",False
4693090,faucetojisan,0.0,False,2023-01-06 17:46:17,Earn crypto by playing fun games online.\nGet ...,"['faucet', 'cointiply', 'BTC']",False


In [ ]:
df.shape

(4693091, 7)

# Text preprocessing

In [ ]:
import re

# --- Các pattern nhận diện bot / tweet tự động ---
BOT_NAME_KEYWORDS = [
    'bot', 'ticker', 'price', 'alert', 'signal', 'tracker',
    'update', 'feed', 'news', 'faucet', 'airdrop'
]

AUTO_TWEET_PATTERNS = [
    r'^\d+\s*\u20bf?\s*=\s*\$[\d,\.]+',   # '1\u20bf = $16814.7...'
    r'^.{0,15}prices?\s+update',           # 'Prices update in $EUR...'
    r'\u2764\ufe0f\s*join me at',          # referral spam
    r'earn crypto by playing',               # faucet spam
    r'join me at bybit',                     # Bybit referral
]

def filter_tweets(df, min_text_length=10, spam_tweet_threshold=50,
                  spam_window_hours=1, report=True):
    """
    Bước 1 — Lọc dữ liệu.

    Các bước thực hiện:
      1. Loại hàng trùng lặp (text + user_name + date)
      2. Loại retweet (dùng cột is_retweet, fallback qua prefix 'RT @')
      3. Loại tweet quá ngắn / quá dài
      4. Loại bot account (detect qua user_name)
      5. Loại tweet tự động / spam (detect qua regex trong text)
      6. Loại spam account (đăng quá nhiều tweet trong 1 cửa sổ giờ)
    """
    df = df.copy()
    initial_count = len(df)
    stats = {}

    # 1. Loại hàng trùng lặp
    before = len(df)
    df = df.drop_duplicates(subset=['text', 'user_name', 'date'])
    stats['duplicates'] = before - len(df)

    # 2. Loại retweet
    before = len(df)
    if 'is_retweet' in df.columns:
        df = df[df['is_retweet'] == False]
    else:
        df = df[~df['text'].str.startswith('RT @', na=False)]
    stats['retweets'] = before - len(df)

    # 3. Loại tweet quá ngắn / quá dài
    before = len(df)
    df['text'] = df['text'].astype(str).str.strip()
    text_len = df['text'].str.len()
    df = df[(text_len >= min_text_length) & (text_len <= 1000)]
    stats['invalid_text'] = before - len(df)

    # 4. Loại bot account dựa vào user_name
    before = len(df)
    bot_pattern = '|'.join(BOT_NAME_KEYWORDS)
    is_bot = df['user_name'].str.lower().str.contains(bot_pattern, na=False)
    df = df[~is_bot]
    stats['bots'] = before - len(df)

    # 5. Loại tweet tự động / spam dựa vào nội dung text
    before = len(df)
    combined_pattern = '|'.join(AUTO_TWEET_PATTERNS)
    is_auto = df['text'].str.lower().str.contains(combined_pattern, na=False, regex=True)
    df = df[~is_auto]
    stats['auto_tweets'] = before - len(df)

    # 6. Loại spam account (đăng quá nhiều tweet trong cửa sổ giờ)
    before = len(df)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['_time_window'] = df['date'].dt.floor(f'{spam_window_hours}h')
    tweet_counts = (
        df.groupby(['user_name', '_time_window'])
          .size()
          .reset_index(name='_count')
    )
    spam_users = tweet_counts[
        tweet_counts['_count'] > spam_tweet_threshold
    ]['user_name'].unique()
    df = df[~df['user_name'].isin(spam_users)].drop(columns=['_time_window'])
    stats['spam_accounts'] = before - len(df)

    # Báo cáo tổng kết
    final_count = len(df)
    total_removed = initial_count - final_count
    retention = (final_count / initial_count) * 100

    if report:
        print('=' * 50)
        print('    BÁO CÁO LỌC DỮ LIỆU — BƯỚC 1')
        print('=' * 50)
        print(f"  Tweet ban đầu         : {initial_count:>10,}")
        print(f"  Trùng lặp loại bỏ     : {stats['duplicates']:>10,}")
        print(f"  Retweet loại bỏ       : {stats['retweets']:>10,}")
        print(f"  Text bất thường       : {stats['invalid_text']:>10,}")
        print(f"  Bot account           : {stats['bots']:>10,}")
        print(f"  Tweet tự động / spam  : {stats['auto_tweets']:>10,}")
        print(f"  Spam account          : {stats['spam_accounts']:>10,}")
        print('-' * 50)
        print(f"  Tổng loại bỏ          : {total_removed:>10,}")
        print(f"  Tweet còn lại         : {final_count:>10,}")
        print(f"  Tỉ lệ giữ lại         : {retention:>9.1f}%")
        print('=' * 50)

    return df.reset_index(drop=True)


df = filter_tweets(df)

    BÁO CÁO LỌC DỮ LIỆU — BƯỚC 1
  Tweet ban đầu         :  4,693,091
  Trùng lặp loại bỏ     :        924
  Retweet loại bỏ       :      4,328
  Text bất thường       :        115
  Bot account           :    466,512
  Tweet tự động / spam  :     34,822
  Spam account          :    381,949
--------------------------------------------------
  Tổng loại bỏ          :    888,650
  Tweet còn lại         :  3,804,441
  Tỉ lệ giữ lại         :      81.1%


In [ ]:
import html

def clean_text(text: str) -> str:
    """
    Bước 2 — Làm sạch nội dung text của một tweet.

    Thứ tự xử lý:
      1. Giải mã HTML entities  (&amp; &lt; &gt; ...)
      2. Xóa URL
      3. Xóa @mention
      4. Xử lý hashtag: tách thành từ thường, giữ nội dung
      5. Xóa ký tự xuống dòng / tab
      6. Xóa ký tự đặc biệt, chỉ giữ chữ + số + dấu câu cơ bản
      7. Chuẩn hóa khoảng trắng

    Lưu ý: KHÔNG lowercase — VADER đọc chữ HOA để tăng cường độ cảm xúc.
    Lưu ý: KHÔNG xóa dấu chấm than / chấm hỏi — VADER dùng để tính score.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    # 1. Giải mã HTML entities: &amp; -> &, &lt; -> <, &#39; -> '
    text = html.unescape(text)

    # 2. Xóa URL (http/https/www)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Xóa @mention
    text = re.sub(r'@\w+', '', text)

    # 4. Xử lý hashtag: #BitcoinMoon -> bitcoin moon, #BTC -> BTC
    #    Tách camelCase/PascalCase: #BitcoinMoon -> Bitcoin Moon
    def expand_hashtag(match):
        tag = match.group(1)
        # Tách PascalCase/camelCase thành các từ riêng
        expanded = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        expanded = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', expanded)
        return expanded

    text = re.sub(r'#(\w+)', expand_hashtag, text)

    # 5. Xóa ký tự xuống dòng, tab, carriage return
    text = re.sub(r'[\r\n\t]+', ' ', text)

    # 6. Chỉ xóa ký tự điều khiển, giữ emoji
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", '', text)

    # 7. Chuẩn hóa khoảng trắng thừa
    text = re.sub(r' {2,}', ' ', text).strip()

    return text


def apply_clean_text(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Áp dụng clean_text() lên toàn bộ cột 'text' của DataFrame.
    Kết quả được lưu vào cột 'text_clean', giữ nguyên cột 'text' gốc.
    """
    df = df.copy()

    df['text_clean'] = df['text'].apply(clean_text)

    # Loại các dòng mà sau khi làm sạch text trở nên rỗng
    before = len(df)
    df = df[df['text_clean'].str.strip().str.len() > 0]
    dropped = before - len(df)

    if report:
        print('=' * 50)
        print('    BÁO CÁO LÀM SẠCH TEXT — BƯỚC 2')
        print('=' * 50)
        print(f'  Tweet đầu vào         : {before:>10,}')
        print(f'  Tweet rỗng sau clean  : {dropped:>10,}')
        print(f'  Tweet còn lại         : {len(df):>10,}')
        print('=' * 50)
        print()
        print('  Ví dụ trước / sau khi làm sạch:')
        samples = df[['text', 'text_clean']].head(3)
        for i, row in samples.iterrows():
            print(f'\n  [{i}] Trước : {row["text"][:80]}')
            print(f'      Sau   : {row["text_clean"][:80]}')
        print('=' * 50)

    return df.reset_index(drop=True)


df = apply_clean_text(df)


    BÁO CÁO LÀM SẠCH TEXT — BƯỚC 2
  Tweet đầu vào         :  3,804,441
  Tweet rỗng sau clean  :          1
  Tweet còn lại         :  3,804,440

  Ví dụ trước / sau khi làm sạch:

  [0] Trước : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.
      Sau   : Blue Ridge Bank shares halted by NYSE after bitcoin ATM announcement …

  [1] Trước : 😎 Today, that's this #Thursday, we will do a "🎬 Take 2" with our friend @LeoWand
      Sau   : 😎 Today, that's this Thursday, we will do a "🎬 Take 2" with our friend , Btc wal

  [2] Trước : Guys evening, I have read this article about BTC and would like to share with yo
      Sau   : Guys evening, I have read this article about BTC and would like to share with yo


In [ ]:
!pip install emoji

In [ ]:
import emoji

def process_hashtags_and_emoji(text: str) -> str:
    """
    Bước 3 — Xử lý Hashtag & Emoji.

    Thứ tự xử lý:
      1. Chuyển emoji thành text mô tả  (🚀 -> 'rocket', 📉 -> 'chart decreasing')
      2. Tách hashtag PascalCase/camelCase thành từ  (#BitcoinMoon -> 'Bitcoin Moon')
         (Bước này bổ sung cho clean_text ở bước 2 — xử lý các hashtag còn sót)
      3. Xóa dấu # còn sót sau khi tách
      4. Chuẩn hóa khoảng trắng

    Lưu ý: Giữ nguyên chữ hoa/thường để VADER đọc được cảm xúc.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    # 1. Chuyển emoji -> dạng :emoji_name: (vd: 🚀 -> ':rocket:')
    # Không dùng delimiters để giữ nguyên dấu hai chấm mặc định
    text = emoji.demojize(text)

    # 2. Tách hashtag PascalCase/camelCase thành từ
    def expand_hashtag(match):
        tag = match.group(1)
        # Tách chữ thường dính chữ hoa: BitcoinMoon -> Bitcoin Moon
        expanded = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        # Tách chữ hoa viết tắt: USDTTo -> USDT To
        expanded = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', expanded)
        return expanded

    text = re.sub(r'#(\w+)', expand_hashtag, text)

    # 3. Xóa dấu # còn sót (không xóa dấu : vì đó là của emoji)
    text = text.replace('#', '')

    # 4. Chuẩn hóa khoảng trắng
    text = re.sub(r' {2,}', ' ', text).strip()

    return text


def apply_process_hashtags_and_emoji(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Áp dụng process_hashtags_and_emoji() lên cột 'text_clean'.
    Kết quả ghi đè trực tiếp lên 'text_clean'.
    """
    df = df.copy()
    df['text_clean'] = df['text_clean'].apply(process_hashtags_and_emoji)

    # Loại dòng rỗng sau xử lý
    before = len(df)
    df = df[df['text_clean'].str.strip().str.len() > 0]
    dropped = before - len(df)

    if report:
        print('=' * 50)
        print('  BÁO CÁO HASHTAG & EMOJI — BƯỚC 3')
        print('=' * 50)
        print(f'  Tweet đầu vào         : {before:>10,}')
        print(f'  Tweet rỗng sau xử lý  : {dropped:>10,}')
        print(f'  Tweet còn lại         : {len(df):>10,}')
        print('=' * 50)
        print()
        print('  Ví dụ trước / sau khi xử lý:')
        # Ưu tiên lấy mẫu có emoji hoặc hashtag
        has_emoji_or_tag = df[
            df['text'].str.contains(r'[^\x00-\x7F]|#\w+', na=False, regex=True)
        ].head(3)
        samples = has_emoji_or_tag if len(has_emoji_or_tag) >= 3 else df.head(3)
        for i, row in samples.iterrows():
            print(f'\n  [{i}] Trước : {row["text"][:80]}')
            print(f'      Sau   : {row["text_clean"][:80]}')
        print('=' * 50)

    return df.reset_index(drop=True)


df = apply_process_hashtags_and_emoji(df)

  BÁO CÁO HASHTAG & EMOJI — BƯỚC 3
  Tweet đầu vào         :  3,804,440
  Tweet rỗng sau xử lý  :          0
  Tweet còn lại         :  3,804,440

  Ví dụ trước / sau khi xử lý:

  [0] Trước : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.
      Sau   : Blue Ridge Bank shares halted by NYSE after bitcoin ATM announcement …

  [1] Trước : 😎 Today, that's this #Thursday, we will do a "🎬 Take 2" with our friend @LeoWand
      Sau   : :smiling_face_with_sunglasses: Today, that's this Thursday, we will do a ":clapp

  [2] Trước : Guys evening, I have read this article about BTC and would like to share with yo
      Sau   : Guys evening, I have read this article about BTC and would like to share with yo


In [ ]:
def normalize_text(text: str) -> str:
    """
    Bước 4 — Chuẩn hóa text.

    Thứ tự xử lý:
      1. Sửa lỗi lặp ký tự quá mức  (soooo -> soo, greaaaat -> greaat)
         Giữ tối đa 2 ký tự giống nhau liên tiếp để VADER vẫn nhận được
         tín hiệu nhấn mạnh (good -> good, goood -> good, gooood -> good)
      2. Chuẩn hóa dấu chấm than / hỏi lặp  (!!!! -> !!, ??? -> ??)
         Giữ tối đa 2 để bảo toàn tín hiệu cảm xúc mà không gây nhiễu
      3. Xóa stopword crypto không mang cảm xúc
         (bitcoin, btc, crypto... chỉ là từ khóa tìm kiếm, không có sentiment)
      4. Xóa token quá ngắn (1 ký tự) còn sót
      5. Chuẩn hóa khoảng trắng

    Lưu ý: KHÔNG lowercase, KHÔNG stemming/lemmatization —
             VADER cần từ nguyên bản để tra từ điển.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    # 1. Sửa ký tự lặp quá mức: gooood -> good, sooo -> soo
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 2. Chuẩn hóa dấu câu lặp: !!! -> !!, ??? -> ??
    text = re.sub(r'!{3,}', '!!', text)
    text = re.sub(r'\?{3,}', '??', text)

    # 3. Xóa stopword crypto (chỉ là từ khóa, không có sentiment)
    CRYPTO_STOPWORDS = {
    'bitcoin', 'btc', 'crypto', 'cryptocurrency', 'blockchain',
    'eth', 'ethereum', 'coin', 'token', 'usd', 'usdt', 'binance',
    }
    tokens = text.split()
    tokens = [t for t in tokens if t.lower() not in CRYPTO_STOPWORDS]

    # 4. Xóa token 1 ký tự còn sót (trừ 'I')
    tokens = [t for t in tokens if len(t) > 1 or t == 'I']

    # 5. Chuẩn hóa khoảng trắng
    text = ' '.join(tokens).strip()

    return text


def apply_normalize_text(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Áp dụng normalize_text() lên cột 'text_clean'.
    Kết quả ghi đè trực tiếp lên 'text_clean'.
    """
    df = df.copy()
    df['text_clean'] = df['text_clean'].apply(normalize_text)

    # Loại dòng rỗng sau chuẩn hóa
    before = len(df)
    df = df[df['text_clean'].str.strip().str.len() > 0]
    dropped = before - len(df)

    if report:
        print('=' * 50)
        print('  BÁO CÁO CHUẨN HÓA TEXT — BƯỚC 4')
        print('=' * 50)
        print(f'  Tweet đầu vào         : {before:>10,}')
        print(f'  Tweet rỗng sau xử lý  : {dropped:>10,}')
        print(f'  Tweet còn lại         : {len(df):>10,}')
        print('=' * 50)
        print()
        print('  Ví dụ trước / sau khi chuẩn hóa:')
        samples = df[['text', 'text_clean']].head(3)
        for i, row in samples.iterrows():
            print(f'\n  [{i}] Trước : {row["text"][:80]}')
            print(f'      Sau   : {row["text_clean"][:80]}')
        print('=' * 50)

    return df.reset_index(drop=True)


df = apply_normalize_text(df)

  BÁO CÁO CHUẨN HÓA TEXT — BƯỚC 4
  Tweet đầu vào         :  3,804,440
  Tweet rỗng sau xử lý  :        379
  Tweet còn lại         :  3,804,061

  Ví dụ trước / sau khi chuẩn hóa:

  [0] Trước : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.
      Sau   : Blue Ridge Bank shares halted by NYSE after ATM announcement

  [1] Trước : 😎 Today, that's this #Thursday, we will do a "🎬 Take 2" with our friend @LeoWand
      Sau   : :smiling_face_with_sunglasses: Today, that's this Thursday, we will do ":clapper

  [2] Trước : Guys evening, I have read this article about BTC and would like to share with yo
      Sau   : Guys evening, I have read this article about and would like to share with you al


In [ ]:
def compute_influence_weight(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Bước 5 — Tính influence weight cho mỗi tweet.

    Công thức:
        weight = log1p(user_favourites) * verified_boost

        Trong đó:
          - log1p(user_favourites): số lượt thích của tài khoản,
            dùng log để tránh tài khoản triệu follower lấn át hoàn toàn
          - verified_boost: 1.5 nếu tài khoản đã xác thực, 1.0 nếu không

    Lưu ý: Dataset không có followers_count nên dùng user_favourites
    (tổng lượt thích tài khoản đã nhận) làm proxy cho mức độ ảnh hưởng.
    """
    df = df.copy()
    #ép kiểu cột user_verified từ str sang bool
    df['user_verified'] = df['user_verified'].map({'True': True, 'False': False, True: True, False: False}).fillna(False)

    # verified_boost: True/False -> 1.5/1.0
    df['verified_boost'] = df['user_verified'].apply(lambda x: 1.5 if x == True else 1.0)

    # log1p(user_favourites) để tránh giá trị 0 và giảm skewness
    df['user_favourites'] = pd.to_numeric(df['user_favourites'], errors='coerce').fillna(0)
    df['weight'] = np.log1p(df['user_favourites']) * df['verified_boost']

    # Nếu weight = 0 (tài khoản không có lượt thích nào và chưa verified)
    # gán weight tối thiểu = 1.0 để tweet vẫn được tính vào aggregate
    df['weight'] = df['weight'].clip(lower=1.0)

    if report:
        print('=' * 50)
        print('  BÁO CÁO INFLUENCE WEIGHT — BƯỚC 5')
        print('=' * 50)
        print(f"  Tài khoản verified    : {df['user_verified'].sum():>10,}")
        print(f"  Tài khoản unverified  : {(~df['user_verified']).sum():>10,}")
        print()
        print('  Phân phối weight:')
        print(f"    min    : {df['weight'].min():>10.2f}")
        print(f"    median : {df['weight'].median():>10.2f}")
        print(f"    mean   : {df['weight'].mean():>10.2f}")
        print(f"    max    : {df['weight'].max():>10.2f}")
        print('=' * 50)
        print()
        print('  Ví dụ 5 tweet có weight cao nhất:')
        top = df[['user_name', 'user_verified', 'user_favourites', 'weight']]\
                .sort_values('weight', ascending=False).head(5)
        print(top.to_string(index=False))
        print('=' * 50)

    return df


df = compute_influence_weight(df)

  BÁO CÁO INFLUENCE WEIGHT — BƯỚC 5
  Tài khoản verified    :     31,198
  Tài khoản unverified  :  3,772,863

  Phân phối weight:
    min    :       1.00
    median :       6.85
    mean   :       6.56
    max    :      21.16

  Ví dụ 5 tweet có weight cao nhất:
           user_name  user_verified  user_favourites    weight
Crypto (DJ) Prestige           True        1337169.0 21.159099
Crypto (DJ) Prestige           True        1269317.0 21.080985
               Perez           True        1226714.0 21.029776
Crypto (DJ) Prestige           True        1076288.0 20.833544
Crypto (DJ) Prestige           True         803786.0 20.395634


#Lưu dataset

In [ ]:
df.to_csv('btc_tweet_38.csv',index=False, encoding='utf-8-sig')

In [ ]:
df0 = pd.read_csv('/kaggle/working/btc_tweet_38.csv',
                chunksize=100000,
                engine='python',
                on_bad_lines='skip')
df = pd.concat(df0, ignore_index=True)

In [ ]:
df.head(8)

,user_name,user_favourites,user_verified,date,text,hashtags,is_retweet,text_clean,verified_boost,weight
0,DeSota Wilson,4838.0,False,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,['bitcoin'],False,Blue Ridge Bank shares halted by NYSE after AT...,1.0,8.484463
1,CryptoND,25483.0,False,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""...","['Thursday', 'Btc', 'wallet', 'security']",False,":smiling_face_with_sunglasses: Today, that's t...",1.0,10.145806
2,Tdlmatias,924.0,False,2021-02-10 23:54:48,"Guys evening, I have read this article about B...",NaN,False,"Guys evening, I have read this article about a...",1.0,6.829794
3,Crypto is the future,14.0,False,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,"['Bitcoin', 'FX', 'BTC', 'crypto']",False,$BTC big chance in billion! Price: \4872644.0 ...,1.0,2.708050
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,10482.0,False,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,['BTC'],False,This network is secured by 508 nodes as of tod...,1.0,9.257510
5,ZerrBenz™ ⚔ ✪ 20732,2444.0,False,2021-02-10 23:53:30,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,"['Crypto', 'Binance', 'Cashback']",False,:chart_increasing_with_yen: Trade on :pushpin:...,1.0,7.801800
6,Mikcoin,238.0,False,2021-02-10 23:52:25,#BTC #Bitcoin #Ethereum #ETH #Crypto #cryptotr...,"['BTC', 'Bitcoin', 'Ethereum', 'ETH', 'Crypto'...",False,cryptotrading $RSR I know told you guys the ta...,1.0,5.476464
7,DeSota Wilson,4838.0,False,2021-02-10 23:52:08,.@Tesla’s #bitcoin investment is revolutionary...,"['bitcoin', 'crypto']",False,.’s investment is revolutionary for but other ...,1.0,8.484463


In [ ]:
# Chia df thành danh sách chứa 10 DataFrame con
df_list = np.array_split(df, 10)

# Truy cập từng df: df_list[0], df_list[1], ..., df_list[9]
# Ví dụ: gán ra các biến riêng lẻ nếu muốn
df1, df2, df3, df4, df5, df6, df7, df8, df9, df10 = df_list

print(f"Số dòng của df1: {len(df1):,}")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Số dòng của df1: 380,407


In [ ]:
print(df1.shape)
print(df2.shape)
print(df3.shape)
print(df4.shape)
print(df5.shape)
print(df6.shape)
print(df7.shape)
print(df8.shape)
print(df9.shape)
print(df10.shape)

(380407, 10)
(380406, 10)
(380406, 10)
(380406, 10)
(380406, 10)
(380406, 10)
(380406, 10)
(380406, 10)
(380406, 10)
(380406, 10)


In [ ]:
df1.to_csv('btc_tweet_1.csv',index=False, encoding='utf-8-sig')
df2.to_csv('btc_tweet_2.csv',index=False, encoding='utf-8-sig')
df3.to_csv('btc_tweet_3.csv',index=False, encoding='utf-8-sig')
df4.to_csv('btc_tweet_4.csv',index=False, encoding='utf-8-sig')
df5.to_csv('btc_tweet_5.csv',index=False, encoding='utf-8-sig')
df6.to_csv('btc_tweet_6.csv',index=False, encoding='utf-8-sig')
df7.to_csv('btc_tweet_7.csv',index=False, encoding='utf-8-sig')
df8.to_csv('btc_tweet_8.csv',index=False, encoding='utf-8-sig')
df9.to_csv('btc_tweet_9.csv',index=False, encoding='utf-8-sig')
df10.to_csv('btc_tweet_10.csv',index=False, encoding='utf-8-sig')